# Offline to online

In [ ]:
group_list = [
    "visual-cube-single-play-singletask-task1-v0",
    "visual-cube-double-play-singletask-task1-v0",
    "visual-puzzle-4x4-play-singletask-task1-v0"
]

In [191]:
import wandb
import pandas as pd

# Project is specified by <entity/project-name>
def get_offon_rundata(filter):
    api = wandb.Api()
    runs = api.runs(
        "dittos-elbows0m-columbia-university/fql", 
        filters=filter
    )
    summary_list, config_list, name_list, run_data = [], [], [], []
    for run in runs:
        df = run.history(keys=["evaluation/success"])
        df = df.rename(columns={'_step': 'Step'})
        df['Step'] = [str(i * 100)+'K' for i in range(0, 10)] + ['1M',]
        df = df.rename(columns={'evaluation/success': 'Success Rate'})
        run_data.append(df)
        # .summary contains the output keys/values for metrics like accuracy.
        #  We call ._json_dict to omit large files
        summary_list.append(run.summary._json_dict)

        # .config contains the hyperparameters.
        #  We remove special values that start with _.
        config_list.append(
            {k: v for k,v in run.config.items()
            if not k.startswith('_')})

        # .name is the human-readable name of the run.
        name_list.append(run.name)
    return run_data

In [194]:
fql_filter = {
    "group": "visual-puzzle-4x4-play-singletask-task1-v0_offon",
    "display_name":  {
        "$in": [
            "sd3456_20260127_030116",
            "sd2345_20260127_030116",
            "sd1234_20260127_030116",
            "sd4567_20260127_030116",
        ]
    },
    "config.agent.agent_name": "fql"
}
run_data = get_offon_rundata(fql_filter)

In [195]:
import os
import json
import matplotlib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

from matplotlib.ticker import FormatStrFormatter
from decimal import Decimal

step_lables = [str(i * 100)+'K' for i in range(0, 10)] + ['1M',]
data = pd.DataFrame({
    'Step': step_lables * 4,
    'Success Rate': np.array([[0, 24, 26, 26, 38, 20, 84, 100, 100, 96, 100],
                    [0, 34, 22, 30, 36, 26, 60, 94, 94, 100, 100],
                    [0, 30, 46, 38, 22, 26, 90, 92, 94, 98, 100],
                    [0, 24, 28, 40, 26, 32, 42, 90, 96, 100, 100]]).flatten(),
})
data['Success Rate'] = data['Success Rate'].astype(float)/100

fql_data = pd.concat(run_data, axis=0)

fe = fm.FontEntry(
    fname=os.path.expanduser('~/prima_serif_roman_bt.ttf'),
    name='primaserif')
fm.fontManager.ttflist.insert(0, fe) # or append is fine
matplotlib.rcParams['font.family'] = fe.name # = 'your custom ttf font name'
palette = sns.color_palette('Set3')

COLORS = [palette[0], palette[2], palette[5], palette[6], palette[8], palette[4], palette[9], palette[7]]

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(8, 7))
ax = sns.lineplot(data=data, x='Step', y='Success Rate', label='Causal-FQL', legend='brief', linewidth=2, color=palette[5])
ax = sns.lineplot(data=fql_data, x='Step', y='Success Rate', label='FQL', legend='brief', linewidth=2, color=COLORS[1])
# ax.legend(labels=['Causal-FQL', 'FQL'])


ax.set_xticks(step_lables[::2], labels=step_lables[::2], fontsize=12, fontname='primaserif')
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], labels=["0", "0.25", "0.5", "0.75", "1.0"], fontsize=12, fontname='primaserif')
ax.set_xlabel('Steps', fontsize=13, labelpad=0, fontname='primaserif')
ax.set_ylabel('Average Success Rate', fontsize=13, labelpad=0, fontname='primaserif')
# Add vertical dashed line at 500K
ax.axvline(x='500K', linestyle='--', color='gray', linewidth=1, alpha=0.4)
# Add "offline" text to the left of the line
ax.text(step_lables[3], 0.02, 'offline', ha='right', va='bottom', fontsize=13, fontname='primaserif', color='gray', rotation=0)
# Add "online" text to the right of the line
ax.text(step_lables[7], 0.02, 'online', ha='left', va='bottom', fontsize=13, fontname='primaserif', color='gray', rotation=0)
ax.legend(fontsize=10, frameon=True)
for text in ax.get_legend().get_texts():
    text.set_fontname('primaserif')

sns.despine(offset=3)
plt.xticks(fontname='primaserif')
plt.yticks(fontname='primaserif')
sns.set_theme(font_scale=10.5)
plt.tight_layout()
plt.title('Visual Puzzle-4x4 Task1', fontsize=13, fontname='primaserif')
# plt.show()
fig = plt.gcf()
fig.set_size_inches(8, 7)
imgPath = f'figures/puzzle4x4_offon.png'
fig.savefig(imgPath, dpi=800, bbox_inches='tight', pad_inches=0)
plt.close()


In [189]:
cube_cfql =  {
    "group": "visual-cube-double-play-singletask-task1-v0_offon",
    "display_name":  {
        "$in": [
            "sd1234_20260128_015549",
            "sd3456_20260128_015549",
            "sd5678_20260128_015549"
        ]
    },
}

cube_fql = {
    "group": "visual-cube-double-play-singletask-task1-v0_offon",
    "display_name":  {
        "$in": [
            "sd4567_20260128_010828",
            "sd3456_20260128_010824",
            "sd2345_20260128_010823",
            "sd1234_20260128_010822"
        ]
    },
    "config.agent.agent_name": "fql",
}

cube_cfql_data = get_offon_rundata(cube_cfql)
cube_fql_data = get_offon_rundata(cube_fql)

In [190]:
# cube-double
import os
import json
import matplotlib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

from matplotlib.ticker import FormatStrFormatter
from decimal import Decimal

step_lables = [str(i * 100)+'K' for i in range(0, 10)] + ['1M',]

cfql_data = pd.concat(cube_cfql_data, axis=0)
fql_data = pd.concat(cube_fql_data, axis=0)

fe = fm.FontEntry(
    fname=os.path.expanduser('~/prima_serif_roman_bt.ttf'),
    name='primaserif')
fm.fontManager.ttflist.insert(0, fe) # or append is fine
matplotlib.rcParams['font.family'] = fe.name # = 'your custom ttf font name'
palette = sns.color_palette('Set3')

COLORS = [palette[0], palette[2], palette[5], palette[6], palette[8], palette[4], palette[9], palette[7]]

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(8, 7))
ax = sns.lineplot(data=cfql_data, x='Step', y='Success Rate', label='Causal-FQL', legend='brief', linewidth=2, color=palette[5])
ax = sns.lineplot(data=fql_data, x='Step', y='Success Rate', label='FQL', legend='brief', linewidth=2, color=COLORS[1])


ax.set_xticks(step_lables[::2], labels=step_lables[::2], fontsize=12, fontname='primaserif')
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], labels=["0", "0.25", "0.5", "0.75", "1.0"], fontsize=12, fontname='primaserif')
ax.set_xlabel('Steps', fontsize=13, labelpad=0, fontname='primaserif')
ax.set_ylabel('Average Success Rate', fontsize=13, labelpad=0, fontname='primaserif')
# Add vertical dashed line at 500K
ax.axvline(x='500K', linestyle='--', color='gray', linewidth=1, alpha=0.4)
# Add "offline" text to the left of the line
ax.text(step_lables[3], 0.02, 'offline', ha='right', va='bottom', fontsize=13, fontname='primaserif', color='gray', rotation=0)
# Add "online" text to the right of the line
ax.text(step_lables[7], 0.02, 'online', ha='left', va='bottom', fontsize=13, fontname='primaserif', color='gray', rotation=0)
ax.legend(fontsize=10, frameon=True)
for text in ax.get_legend().get_texts():
    text.set_fontname('primaserif')

sns.despine(offset=3)
plt.xticks(fontname='primaserif')
plt.yticks(fontname='primaserif')
sns.set_theme(font_scale=10.5)
plt.tight_layout()
plt.title('Visual Cube-Double Task1', fontsize=13, fontname='primaserif')
# plt.show()
fig = plt.gcf()
fig.set_size_inches(8, 7)
imgPath = f'figures/cube_double_offon.png'
fig.savefig(imgPath, dpi=800, bbox_inches='tight', pad_inches=0)
plt.close()


In [187]:
cube_single_cfql =  {
    "group": "visual-cube-single-play-singletask-task1-v0_offon",
    "display_name":  {
        "$in": [
            "sd4567_20260128_015549",
            "sd2345_20260128_010141",
            "sd3456_20260128_010141",
            # "sd1234_20260128_143730",
            # "sd5678_20260128_015549"
        ]
    },
}

cube_single_fql = {
    "group": "visual-cube-single-play-singletask-task1-v0_offon",
    "display_name":  {
        "$in": [
            "sd4567_20260128_010142",
            "sd2345_20260128_010141",
            "sd3456_20260128_010141",
            "sd1234_20260128_010141"
        ]
    },
    "config.agent.agent_name": "fql",
}

cube_single_cfql_data = get_offon_rundata(cube_single_cfql)
cube_single_fql_data = get_offon_rundata(cube_single_fql)

In [188]:
# cube-single
import os
import json
import matplotlib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

from matplotlib.ticker import FormatStrFormatter
from decimal import Decimal

step_lables = [str(i * 100)+'K' for i in range(0, 10)] + ['1M',]

cfql_data = pd.concat(cube_single_cfql_data, axis=0)
fql_data = pd.concat(cube_single_fql_data, axis=0)

fe = fm.FontEntry(
    fname=os.path.expanduser('~/prima_serif_roman_bt.ttf'),
    name='primaserif')
fm.fontManager.ttflist.insert(0, fe) # or append is fine
matplotlib.rcParams['font.family'] = fe.name # = 'your custom ttf font name'
palette = sns.color_palette('Set3')

COLORS = [palette[0], palette[2], palette[5], palette[6], palette[8], palette[4], palette[9], palette[7]]

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(8, 7))
ax = sns.lineplot(data=cfql_data, x='Step', y='Success Rate', label='Causal-FQL', legend='brief', linewidth=2, color=palette[5])
ax = sns.lineplot(data=fql_data, x='Step', y='Success Rate', label='FQL', legend='brief', linewidth=2, color=COLORS[1])


ax.set_xticks(step_lables[::2], labels=step_lables[::2], fontsize=12, fontname='primaserif')
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], labels=["0", "0.25", "0.5", "0.75", "1.0"], fontsize=12, fontname='primaserif')
ax.set_xlabel('Steps', fontsize=13, labelpad=0, fontname='primaserif')
ax.set_ylabel('Average Success Rate', fontsize=13, labelpad=0, fontname='primaserif')
# Add vertical dashed line at 500K
ax.axvline(x='500K', linestyle='--', color='gray', linewidth=1, alpha=0.4)
# Add "offline" text to the left of the line
ax.text(step_lables[3], 0.02, 'offline', ha='right', va='bottom', fontsize=13, fontname='primaserif', color='gray', rotation=0)
# Add "online" text to the right of the line
ax.text(step_lables[7], 0.02, 'online', ha='left', va='bottom', fontsize=13, fontname='primaserif', color='gray', rotation=0)
ax.legend(fontsize=10, frameon=True)
for text in ax.get_legend().get_texts():
    text.set_fontname('primaserif')

sns.despine(offset=3)
plt.xticks(fontname='primaserif')
plt.yticks(fontname='primaserif')
sns.set_theme(font_scale=10.5)
plt.tight_layout()
plt.title('Visual Cube-Single Task1', fontsize=13, fontname='primaserif')
# plt.show()
fig = plt.gcf()
fig.set_size_inches(8, 7)
imgPath = f'figures/cube_single_offon.png'
fig.savefig(imgPath, dpi=800, bbox_inches='tight', pad_inches=0)
plt.close()


## Offline

In [133]:
import wandb
import pandas as pd
api = wandb.Api()

def get_run_data(filters):
    runs = api.runs(
        "dittos-elbows0m-columbia-university/fql", 
        filters=filters
    )
    summary_list, config_list, name_list, run_data = [], [], [], []
    for run in runs:
        df = run.history(keys=["evaluation/success"])
        df = df.rename(columns={'_step': 'Step'})
        df['Step'] = [str(i * 100)+'K' for i in range(0, 6)]
        df = df.rename(columns={'evaluation/success': 'Success Rate'})
        run_data.append(df)
        # .summary contains the output keys/values for metrics like accuracy.
        #  We call ._json_dict to omit large files
        summary_list.append(run.summary._json_dict)

        # .config contains the hyperparameters.
        #  We remove special values that start with _.
        config_list.append(
            {k: v for k,v in run.config.items()
            if not k.startswith('_')})

        # .name is the human-readable name of the run.
        name_list.append(run.name)
    return run_data

In [146]:
filters_disc_5 = {
    "display_name":  {
        "$in": [
            # "sd2345_20260110_114536_ours_en_disc_5.0", 
            # "sd1234_20260110_114540", 
            "sd3456_20260110_114552", 
            "sd4567_20260110_114600"
            "sd62849_20260110_192759_ours_en_disc_5.0",
            "sd37595_20260110_192808"
        ]
    },
}

filters_disc_10_decay4 = {
    "display_name":  {
        "$in": [
            "sd2345_20260111_212248_ours_en_exp_disc_10.0_decay_4", 
            "sd1234_20260111_212313", 
            "sd3456_20260111_212323", 
            "sd4567_20260111_212333"
        ]
    },
}

filters_disc_10_decay2 = {
    "display_name":  {
        "$in": [
            "sd1234_20260112_163003_ours_en_exp_disc_10.0_decay_2", 
            "sd2345_20260112_163006", 
            "sd4567_20260112_163112", 
            "sd5678_20260112_163321"
        ]
    },
}

run_data_disc_5 = pd.concat(get_run_data(filters_disc_5), axis=0)
run_data_disc_10_decay4 = pd.concat(get_run_data(filters_disc_10_decay4), axis=0)
run_data_disc_10_decay2 = pd.concat(get_run_data(filters_disc_10_decay2), axis=0)

In [138]:
filters_disc_15_decay2 = {
    "display_name":  {
        "$in": [
            "sd1234_20260113_003144_ours_en_exp_disc_15.0_decay_2", 
            "sd3456_20260113_003401", 
            "sd5678_20260113_003504", 
            "sd891011_20260114_033856"
        ]
    },
}
run_data_disc_15_decay2 = pd.concat(get_run_data(filters_disc_15_decay2), axis=0)

In [152]:
filters_disc_1 = {
    "display_name":  {
        "$in": [
            "sd2345_20260110_035058", 
            "sd62748_20260110_034748", 
            "sd4567_20260110_034724", 
            "sd3456_20260110_034712",
            "sd1234_20260109_233312_ours_en"
        ]
    },
}
run_data_disc_1 = pd.concat(get_run_data(filters_disc_1), axis=0)

In [165]:
import os
import json
import matplotlib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

from matplotlib.ticker import FormatStrFormatter
from decimal import Decimal

step_lables = [str(i * 100)+'K' for i in range(0, 6)]

fe = fm.FontEntry(
    fname=os.path.expanduser('~/prima_serif_roman_bt.ttf'),
    name='primaserif')
fm.fontManager.ttflist.insert(0, fe) # or append is fine
matplotlib.rcParams['font.family'] = fe.name # = 'your custom ttf font name'
palette = sns.color_palette('Set3')

COLORS = [palette[0], palette[2], palette[5], palette[6], palette[8], palette[4], palette[9], palette[7]]

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(8, 7))
ax = sns.lineplot(data=run_data_disc_1, x='Step', y='Success Rate', label='disc. coef=1.0', legend='brief', linewidth=2, color=palette[6])
ax = sns.lineplot(data=run_data_disc_5, x='Step', y='Success Rate', label='disc. coef=5.0', legend='brief', linewidth=2, color=palette[0])
# ax = sns.lineplot(data=run_data_disc_10_decay2, x='Step', y='Success Rate', label='disc. coef=10.0', legend='brief', linewidth=2, color=palette[1])
ax = sns.lineplot(data=run_data_disc_10_decay4, x='Step', y='Success Rate', label='disc. coef=10.0', legend='brief', linewidth=2, color=palette[2])
ax = sns.lineplot(data=run_data_disc_15_decay2, x='Step', y='Success Rate', label='disc. coef=15.0', legend='brief', linewidth=2, color=palette[3])


ax.set_xticks(step_lables, labels=step_lables, fontsize=12, fontname='primaserif')
ax.set_yticks([0, 0.25, 0.5, 0.75], labels=["0", "0.25", "0.5", "0.75"], fontsize=12, fontname='primaserif')
ax.set_xlabel('Steps', fontsize=13, labelpad=0, fontname='primaserif')
ax.set_ylabel('Average Success Rate', fontsize=13, labelpad=0, fontname='primaserif')
ax.legend(fontsize=10, frameon=True)
for text in ax.get_legend().get_texts():
    text.set_fontname('primaserif')

sns.despine(offset=3)
plt.xticks(fontname='primaserif')
plt.yticks(fontname='primaserif')
sns.set_theme(font_scale=12)
plt.tight_layout()
plt.title('Visual-Cube-Double-Task1', fontsize=13, fontname='primaserif')
# plt.show()
fig = plt.gcf()
fig.set_size_inches(8, 7)
imgPath = f'figures/discdecay.png'
fig.savefig(imgPath, dpi=800, bbox_inches='tight', pad_inches=0)
plt.close()


## Ensembles

In [185]:
filters_en2 = {
    "display_name":  {
        "$in": [
            "sd1234_20260113_003144_ours_en_exp_disc_15.0_decay_2", 
            "sd3456_20260113_003401", 
            "sd5678_20260113_003504", 
            "sd891011_20260114_033856"
        ]
    },
}

# filters_en5 = {
#     "display_name":  {
#         "$in": [
#             "sd1234_20260110_114720_ours_en_5", 
#             "sd3456_20260110_114727", 
#             "sd4567_20260110_114733", 
#         ]
#     },
# }

filters_en4 = {
    "display_name":  {
        "$in": [
            "sd1234_20260128_145407", 
            "sd3456_20260128_145407", 
            "sd4567_20260128_145407", 
            "sd2345_20260128_145407"
        ]
    },
    "config.agent.num_ensembles": 4,
}

filters_en6 = {
    "display_name":  {
        "$in": [
            "sd3456_20260128_145407", 
            "sd2345_20260128_145407", 
            "sd4567_20260128_145407", 
            "sd1234_20260128_145407"
        ]
    },
    "config.agent.num_ensembles": 6,
}
run_data_en2 = pd.concat(get_run_data(filters_en2), axis=0)
# run_data_en5 = pd.concat(get_run_data(filters_en5), axis=0)
run_data_en4 = pd.concat(get_run_data(filters_en4), axis=0)
run_data_en6 = pd.concat(get_run_data(filters_en6), axis=0)

In [208]:
import os
import json
import matplotlib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

from matplotlib.ticker import FormatStrFormatter
from decimal import Decimal

step_lables = [str(i * 100)+'K' for i in range(0, 6)]

fe = fm.FontEntry(
    fname=os.path.expanduser('~/prima_serif_roman_bt.ttf'),
    name='primaserif')
fm.fontManager.ttflist.insert(0, fe) # or append is fine
matplotlib.rcParams['font.family'] = fe.name # = 'your custom ttf font name'
palette = sns.color_palette('Set3')

COLORS = [palette[0], palette[2], palette[5], palette[6], palette[8], palette[4], palette[9], palette[7]]

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(8, 7))
ax = sns.lineplot(data=run_data_en2, x='Step', y='Success Rate', label='num_ensemble=2', legend='brief', linewidth=2, color=palette[6])
ax = sns.lineplot(data=run_data_en4, x='Step', y='Success Rate', label='num_ensemble=4', legend='brief', linewidth=2, color=palette[5])
ax = sns.lineplot(data=run_data_en6, x='Step', y='Success Rate', label='num_ensemble=6', legend='brief', linewidth=2, color=palette[2])


ax.set_xticks(step_lables, labels=step_lables, fontsize=12, fontname='primaserif')
ax.set_yticks([0, 0.25, 0.5, 0.75], labels=["0", "0.25", "0.5", "0.75"], fontsize=12, fontname='primaserif')
ax.set_xlabel('Steps', fontsize=13, labelpad=0, fontname='primaserif')
ax.set_ylabel('Average Success Rate', fontsize=13, labelpad=0, fontname='primaserif')
ax.legend(fontsize=10, frameon=True)
for text in ax.get_legend().get_texts():
    text.set_fontname('primaserif')

sns.despine(offset=3)
plt.xticks(fontname='primaserif')
plt.yticks(fontname='primaserif')
sns.set_theme(font_scale=12)
plt.tight_layout()
plt.title('Visual-Cube-Double-Task1', fontsize=13, fontname='primaserif')
# plt.show()
fig = plt.gcf()
fig.set_size_inches(8,7)
imgPath = f'figures/numen.png'
fig.savefig(imgPath, dpi=800, bbox_inches='tight', pad_inches=0)
plt.close()


## Env Vis

In [206]:
import ogbench
import matplotlib.pyplot as plt
import numpy as np
import os

# To visualize goal, need to change cube_env.py: initialize_episode, 
# move render goal part after second initialize_arm and also comment out self._data.qpos, qvel reset part
# Also need to change resolution in manipspace.__init__

os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "true"
for env_name in ["visual-cube-single-play-singletask-task3-v0", 
                 "visual-cube-double-play-singletask-task4-v0", 
                 "visual-puzzle-3x3-play-singletask-task2-v0",
                 "visual-puzzle-4x4-play-singletask-task3-v0",
                 "visual-scene-play-singletask-task4-v0"
                 ]:
    eval_env = ogbench.make_env_and_datasets(env_name, env_only=True, pixel_transparent_arm=False)
    obs, info = eval_env.reset(seed=100)
    # goal_img = info["goal_rendered"]
    # plt.imsave(f"{env_name}_goal_low.png", goal_img, dpi=500)
    plt.imsave(f"figures/{env_name}_high.png", obs, dpi=500)
    # print(obs)
    eval_env.close()

## Value flow

In [ ]:
import os
import json
import matplotlib
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm

from matplotlib.ticker import FormatStrFormatter
from decimal import Decimal

step_lables = [str(i * 100)+'K' for i in range(0, 5)]

fe = fm.FontEntry(
    fname=os.path.expanduser('~/prima_serif_roman_bt.ttf'),
    name='primaserif')
fm.fontManager.ttflist.insert(0, fe) # or append is fine
matplotlib.rcParams['font.family'] = fe.name # = 'your custom ttf font name'
palette = sns.color_palette('Set3')

COLORS = [palette[0], palette[2], palette[5], palette[6], palette[8], palette[4], palette[9], palette[7]]

cvflow_data = pd.DataFrame({
    'Step': step_lables,
    'Success Rate': np.array([[0, 6, 22, 26, 62,],
                    # [0, 34, 22, 30, 36, 26, 60, 94, 94, 100, 100],
                    # [0, 30, 46, 38, 22, 26, 90, 92, 94, 98, 100],
                    # [0, 24, 28, 40, 26, 32, 42, 90, 96, 100, 100]
                ]).flatten()/100,
})

sns.reset_defaults()
fig, ax = plt.subplots(figsize=(8, 7))
ax = sns.lineplot(data=cvflow_data, x='Step', y='Success Rate', label='Causal Value Flows', legend='brief', linewidth=2, color=palette[6])
ax.axhline(y=0.35, linestyle='--', color=palette[5], linewidth=1, alpha=0.6)
ax.text(step_lables[0], 0.37, 'Value Flows @ 1M steps: 0.37', ha='left', va='bottom', fontsize=8, fontname='primaserif', color=palette[5])

ax.axhline(y=0.62, linestyle='--', color=palette[6], linewidth=1)
ax.text(step_lables[0], 0.64, 'Causal Value Flows @ 400K steps: 0.62', ha='left', va='bottom', fontsize=8, fontname='primaserif', color=palette[6])

ax.set_xticks(step_lables, labels=step_lables, fontsize=12, fontname='primaserif')
ax.set_yticks([0, 0.25, 0.5, 0.75], labels=["0", "0.25", "0.5", "0.75"], fontsize=12, fontname='primaserif')
ax.set_xlabel('Steps', fontsize=13, labelpad=0, fontname='primaserif')
ax.set_ylabel('Average Success Rate', fontsize=13, labelpad=0, fontname='primaserif')
# ax.legend(fontsize=10, frameon=True)
# for text in ax.get_legend().get_texts():
    # text.set_fontname('primaserif')

sns.despine(offset=3)
plt.xticks(fontname='primaserif')
plt.yticks(fontname='primaserif')
sns.set_theme(font_scale=12)
plt.tight_layout()
plt.title('Visual-Cube-Double-Task1', fontsize=13, fontname='primaserif')
# plt.show()
fig = plt.gcf()
fig.set_size_inches(8,7)
imgPath = f'figures/vflow.png'
fig.savefig(imgPath, dpi=800, bbox_inches='tight', pad_inches=0)
plt.close()
